# Aboveground Carbon Across Biomes

Compares aboveground carbon density (Mg C/ha) across four contrasting forest types using ESA CCI Biomass, restricted to forest-cover pixels identified via unsupervised clustering of AlphaEarth satellite embeddings (cross-referenced with Hansen tree cover to label the forest cluster). Also compares the four zones structurally using their forest-only AlphaEarth embedding signatures.

**Zones:** boreal managed forest (Abitibi, Quebec), intact tropical rainforest (Tapajos, Brazil), native temperate forest (Alerce Costero, Chile), and an even-aged Pinus radiata plantation (Biobio, Chile).

**Note on GEDI:** an earlier version of this notebook cross-checked ESA CCI against GEDI L4B (1 km gridded biomass). That comparison was dropped after diagnostics showed it was unreliable at this AOI scale: GEDI L4B's `MU` band is defined as the mean biomass 'including forest and non-forest' within each 1 km cell (not a forest-only value, so partial-cover cells can't be corrected after the fact by masking), and real GEDI sampling density varied sharply across zones (e.g. Tapajos averaged under 1 ground track per 1 km cell, meaning most of its 'data' was a statistical fill-in rather than a direct measurement). See the README for the diagnostic numbers.

**Pipeline:** define zones -> AlphaEarth clustering + forest-mask identification -> carbon from ESA CCI Biomass (forest-weighted) -> comparison chart -> forest-only AlphaEarth signatures -> similarity heatmap.

In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from zones import ZONES, get_zone_geometry, get_zone_label
from carbon_sources import get_esa_cci_carbon, forest_weighted_mean_carbon
from embedding_utils import get_mean_embedding, cosine_similarity_matrix, cluster_and_identify_forest

PROJECT = "your-gee-project-id"
ee.Initialize(project=PROJECT)

## 1. Study zones

Small bounding boxes (~20-25 km) — illustrative placeholders centered on well-known sites for each forest type. Adjust in `src/zones.py` if you have more precise boundaries.

In [ ]:
for key in ZONES:
    geom = get_zone_geometry(key)
    area_ha = geom.area().divide(10000).getInfo()
    print(f"{key}: {get_zone_label(key)} — {area_ha:,.0f} ha")

## 2. Forest mask per zone (AlphaEarth clustering + Hansen tree cover)

A raw bounding box mixes forest with roads, water, clearings, and secondary cover, biasing the carbon average. This runs unsupervised k-means on the AlphaEarth embedding for each zone, then labels whichever cluster has the highest mean Hansen tree-cover (2000) as 'forest' and keeps only those pixels for everything below — both the carbon calculation and the embedding signature used for the similarity heatmap.

In [ ]:
CLUSTER_YEAR = 2023
N_CLUSTERS = 4

forest_masks = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    mask, treecover_by_cluster, forest_id = cluster_and_identify_forest(
        CLUSTER_YEAR, geom, n_clusters=N_CLUSTERS
    )
    forest_masks[key] = mask
    print(f"{key}: cluster {forest_id} selected as forest "
          f"(mean tree cover by cluster: {treecover_by_cluster})")

## 3. Aboveground carbon — ESA CCI Biomass (2022), forest-weighted

Continuous, gap-free 100 m maps. Citation: Santoro & Cartus (2025), ESA CCI Biomass v6.0. Each 10 m forest pixel contributes the carbon value of the coarser cell it falls in, weighting proportionally by forest coverage rather than a hard threshold.

In [ ]:
YEAR = 2022  # most recent year available in ESA CCI Biomass v6.0

esa_results = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    esa_raw = get_esa_cci_carbon(YEAR, geom)
    mean_carbon = forest_weighted_mean_carbon(esa_raw, forest_masks[key], geom, scale=10)
    esa_results[key] = mean_carbon
    print(f"{key}: {mean_carbon:,.1f} Mg C/ha (ESA CCI Biomass, forest-weighted)")

## 4. Result 1: aboveground carbon across biomes

The headline chart for the LinkedIn post.

In [ ]:
comparison_df = pd.DataFrame({
    "zone": [get_zone_label(k) for k in ZONES],
    "ESA CCI Biomass": [esa_results[k] for k in ZONES],
})
comparison_df.to_csv("../figures/carbon_comparison.csv", index=False)
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["darkgreen", "orangered", "steelblue", "goldenrod"]

ax.bar(comparison_df["zone"], comparison_df["ESA CCI Biomass"], color=colors)
ax.set_ylabel("Aboveground carbon (Mg C/ha)")
ax.set_title("Aboveground carbon across biomes — ESA CCI Biomass, forest-weighted")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("../figures/carbon_comparison.png", dpi=200)
plt.show()

## 5. AlphaEarth zone signatures (forest pixels only)

A complementary structural check: how distinct are these four sites in AlphaEarth's 64-dimensional embedding space, once restricted to the same forest mask used for carbon? An earlier version of this used the raw bounding-box mean, which was dominated by regional climate/geography signal (e.g. two nearby Chilean zones looked nearly identical) rather than forest structure — masking to forest-only pixels isolates the structural signal instead.

In [ ]:
EMBEDDING_YEAR = 2023

embedding_img_by_zone = {}
embeddings = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    full_embedding = (
        ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
        .filterDate(f"{EMBEDDING_YEAR}-01-01", f"{EMBEDDING_YEAR + 1}-01-01")
        .filterBounds(geom)
        .mosaic()
        .updateMask(forest_masks[key])
    )
    band_names = full_embedding.bandNames().getInfo()
    stats = full_embedding.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=geom, scale=10, maxPixels=1e13, bestEffort=True,
    ).getInfo()
    embeddings[key] = np.array([stats.get(b, 0.0) or 0.0 for b in band_names])
    print(f"{key}: forest-only embedding vector computed ({len(embeddings[key])} dims)")

## 6. Result 2: zone similarity heatmap for LinkedIn

In [ ]:
labels, sim_matrix = cosine_similarity_matrix(embeddings)
display_labels = [get_zone_label(k) for k in labels]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(sim_matrix, cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(display_labels, rotation=30, ha="right")
ax.set_yticklabels(display_labels)
ax.set_title("AlphaEarth embedding similarity between zones (forest pixels only)")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                 color="white" if sim_matrix[i, j] > 0.5 else "black")

plt.colorbar(im, label="Cosine similarity")
plt.tight_layout()
plt.savefig("../figures/zone_similarity_heatmap.png", dpi=200)
plt.show()

## 7. Combined result for LinkedIn

Headline numbers for the post caption.

In [ ]:
for key in ZONES:
    print(f"{get_zone_label(key)}: {esa_results[key]:,.1f} Mg C/ha (ESA CCI Biomass)")